In [3]:


#Load the Dataset

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("diabetes.csv")
print("Dataset shape:", df.shape)
print(df.head())


# Separate Features and Target

X = df.drop(columns="Outcome")
y = df["Outcome"]

print("Features:", X.shape, "Target:", y.shape)


# Split Training and Test Data

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)


#  Missing Values and Scaling

zero_columns = [
    "Glucose", "BloodPressure",
    "SkinThickness", "Insulin", "BMI"
]

other_columns = [
    col for col in X.columns
    if col not in zero_columns
]

print("Invalid zeros in training data:")
print((X_train[zero_columns] == 0).sum())

zero_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        missing_values=0, strategy="median"
    )),
    ("scaler", StandardScaler())
])

other_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("zero_as_missing", zero_pipeline, zero_columns),
    ("other_features", other_pipeline, other_columns)
])


#  Apply Preprocessing and Verify

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print(
    "Missing values remaining:",
    np.isnan(X_train_processed).sum()
    + np.isnan(X_test_processed).sum()
)

assert X_train_processed.shape == (len(X_train), 8)
assert X_test_processed.shape == (len(X_test), 8)
assert (
    np.isfinite(X_train_processed).all()
    and np.isfinite(X_test_processed).all()
)


# Save Files 

joblib.dump(preprocessor, "preprocessing_pipeline.joblib")

X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Saved pipeline and four split CSV files.")


Dataset shape: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
Features: (768, 8) Target: (768,)
Training: (614, 8)
Testing: (154, 8)
Invalid zeros in training data:
Glucose            4
BloodPressure     23
SkinThickness    175
Insulin          290
BMI                9
dtype: int64
Processed training shape: (614, 8)
Processed testing shape